# Spectral Gap Correlation & Cross-Experiment Analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import yaml
import pickle

from src.model import InstrumentedGPS
from src.datasets import get_dataloaders, get_datasets, DATASET_INFO, compute_spectral_properties
from src.metrics import compute_sink_scores
from src.diagnostics import load_diagnostics, aggregate_metrics
from src.plotting import plot_spectral_gap_correlation, plot_layer_diagnostics, plot_vnode_comparison

## 1. Per-graph spectral gap vs sink rate (H5)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

experiment_id = 'zinc-novnode-rwse-10L'
config_path = f'../outputs/{experiment_id}/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

dataset_info = DATASET_INFO[config['data']['dataset']].copy()
_, _, test_loader, _ = get_dataloaders(config)

model = InstrumentedGPS(config, dataset_info).to(device)
model.load_state_dict(torch.load(f'../outputs/{experiment_id}/best_model.pt', map_location=device, weights_only=True))
model.eval()
model._register_attn_hooks()

# Also load raw test dataset (without transforms) for spectral gap computation
from torch_geometric.datasets import ZINC
test_dataset_raw = ZINC(root='../data/ZINC', subset=True, split='test')

print(f"Loaded model and {len(test_dataset_raw)} test graphs")

In [ ]:
from tqdm import tqdm

per_graph_sink_rates = []
per_graph_lambda2 = []
per_graph_num_nodes = []

max_graphs = 200
graphs_done = 0
raw_graph_idx = 0

for batch in tqdm(test_loader, desc='Computing per-graph metrics'):
    if graphs_done >= max_graphs:
        break
    
    batch = batch.to(device)
    with torch.no_grad():
        _ = model(batch, collect_diagnostics=True)
    
    batch_ids = model.layer_data[0]['batch']
    unique_graphs = batch_ids.unique()
    
    for g_idx, g_id in enumerate(unique_graphs):
        if graphs_done >= max_graphs:
            break
        
        graph_mask = (batch_ids == g_id)
        num_nodes_g = graph_mask.sum().item()
        
        # Sink rate at last layer
        last_layer_idx = model.num_layers - 1
        attn_full = model.attn_weights[last_layer_idx]
        if attn_full is not None and g_idx < attn_full.size(0):
            attn_g = attn_full[g_idx, :, :num_nodes_g, :num_nodes_g]
            sink_data = compute_sink_scores(attn_g)
            per_graph_sink_rates.append(sink_data['overall_sink_rate'])
        else:
            per_graph_sink_rates.append(np.nan)
        
        # Spectral gap from raw graph
        if raw_graph_idx < len(test_dataset_raw):
            spec = compute_spectral_properties(test_dataset_raw[raw_graph_idx])
            per_graph_lambda2.append(spec['lambda_2'])
            per_graph_num_nodes.append(spec['num_nodes'])
        else:
            per_graph_lambda2.append(np.nan)
            per_graph_num_nodes.append(np.nan)
        
        raw_graph_idx += 1
        graphs_done += 1

model._remove_attn_hooks()

per_graph_sink_rates = np.array(per_graph_sink_rates)
per_graph_lambda2 = np.array(per_graph_lambda2)
per_graph_num_nodes = np.array(per_graph_num_nodes)

# Filter NaN
valid = ~np.isnan(per_graph_sink_rates) & ~np.isnan(per_graph_lambda2)
print(f"Valid graphs: {valid.sum()}/{len(valid)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel 1: lambda_2 vs sink rate
r, p = spearmanr(per_graph_lambda2[valid], per_graph_sink_rates[valid])
axes[0].scatter(per_graph_lambda2[valid], per_graph_sink_rates[valid], alpha=0.3, s=15, color='steelblue')
axes[0].set_xlabel('Spectral gap ($\\lambda_2$)')
axes[0].set_ylabel('Per-graph sink rate (last layer)')
axes[0].set_title(f'$\\lambda_2$ vs Sink Rate (r={r:.3f}, p={p:.2e})')
axes[0].grid(True, alpha=0.3)

# Panel 2: num_nodes vs sink rate
r2, p2 = spearmanr(per_graph_num_nodes[valid], per_graph_sink_rates[valid])
axes[1].scatter(per_graph_num_nodes[valid], per_graph_sink_rates[valid], alpha=0.3, s=15, color='darkorange')
axes[1].set_xlabel('Number of nodes')
axes[1].set_ylabel('Per-graph sink rate (last layer)')
axes[1].set_title(f'Graph size vs Sink Rate (r={r2:.3f}, p={p2:.2e})')
axes[1].grid(True, alpha=0.3)

# Panel 3: lambda_2 vs num_nodes (context)
axes[2].scatter(per_graph_num_nodes[valid], per_graph_lambda2[valid], alpha=0.3, s=15, color='tab:green')
axes[2].set_xlabel('Number of nodes')
axes[2].set_ylabel('Spectral gap ($\\lambda_2$)')
axes[2].set_title('Graph Size vs Spectral Gap')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Figure 5: Spectral Gap Predicts Sink Strength (H5)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figure5_spectral_gap_correlation.pdf', bbox_inches='tight', dpi=150)
plt.show()

print(f"\nH5 test: lambda_2 vs sink rate")
print(f"  Spearman r = {r:.3f}, p = {p:.2e}")
print(f"  {'SUPPORTED' if p < 0.05 else 'NOT SUPPORTED'} at p < 0.05")

## 2. Cross-experiment comparison (ablation overview)

In [ ]:
# Try to load diagnostics from all experiments
experiment_ids = [
    'zinc-vnode-rwse-10L', 'zinc-novnode-rwse-10L',
    'zinc-vnode-lappe-10L', 'zinc-vnode-nope-10L',
    'zinc-vnode-rwse-4L', 'zinc-vnode-rwse-16L',
    'zinc-vnode-rwse-10L-nompnn', 'zinc-novnode-rwse-10L-nompnn',
]

results = {}
for eid in experiment_ids:
    diag_path = f'../outputs/{eid}/diagnostics.pkl'
    config_path = f'../outputs/{eid}/config.yaml'
    if os.path.exists(diag_path) and os.path.exists(config_path):
        with open(config_path) as f:
            cfg = yaml.safe_load(f)
        metrics = load_diagnostics(diag_path)
        agg, layers = aggregate_metrics(metrics)
        
        results[eid] = {
            'vnode': cfg['vnode']['enabled'],
            'pe': cfg['pe']['type'],
            'layers': cfg['architecture']['num_layers'],
            'mpnn': cfg['architecture']['mpnn'],
            'max_sink_score': np.nanmax(agg['max_sink_score']['mean']) if 'max_sink_score' in agg else None,
            'max_sink_rate': np.nanmax(agg['overall_sink_rate']['mean']) if 'overall_sink_rate' in agg else None,
            'max_norm_ratio': np.nanmax(agg['max_to_mean_ratio']['mean']) if 'max_to_mean_ratio' in agg else None,
            'min_entropy': np.nanmin(agg['matrix_entropy']['mean']) if 'matrix_entropy' in agg else None,
        }
        print(f"  Loaded: {eid}")
    else:
        print(f"  Missing: {eid}")

print(f"\nLoaded {len(results)}/{len(experiment_ids)} experiments")

In [ ]:
print(f"\n{'Experiment':<35} {'VNode':>6} {'PE':>6} {'Layers':>7} {'MPNN':>6} {'MaxSink':>8} {'SinkRate':>9} {'NormRat':>8} {'MinEnt':>8}")
print("=" * 110)

for eid, r in sorted(results.items()):
    vn = 'ON' if r['vnode'] else 'OFF'
    ms = f"{r['max_sink_score']:.4f}" if r['max_sink_score'] is not None else 'N/A'
    sr = f"{r['max_sink_rate']:.4f}" if r['max_sink_rate'] is not None else 'N/A'
    nr = f"{r['max_norm_ratio']:.2f}" if r['max_norm_ratio'] is not None else 'N/A'
    me = f"{r['min_entropy']:.4f}" if r['min_entropy'] is not None else 'N/A'
    print(f"{eid:<35} {vn:>6} {r['pe']:>6} {r['layers']:>7} {r['mpnn']:>6} {ms:>8} {sr:>9} {nr:>8} {me:>8}")

## 3. VNode vs No-VNode layer-wise comparison

In [ ]:
# Load both diagnostics
vnode_diag = load_diagnostics('../outputs/zinc-vnode-rwse-10L/diagnostics.pkl') if os.path.exists('../outputs/zinc-vnode-rwse-10L/diagnostics.pkl') else None
novnode_diag = load_diagnostics('../outputs/zinc-novnode-rwse-10L/diagnostics.pkl') if os.path.exists('../outputs/zinc-novnode-rwse-10L/diagnostics.pkl') else None

if vnode_diag and novnode_diag:
    agg_vn, layers_vn = aggregate_metrics(vnode_diag)
    agg_novn, layers_novn = aggregate_metrics(novnode_diag)
    
    metrics_to_compare = [
        ('max_sink_score', 'Max Sink Score'),
        ('overall_sink_rate', 'Overall Sink Rate'),
        ('matrix_entropy', 'Matrix Entropy'),
        ('max_to_mean_ratio', 'Max/Mean Norm Ratio'),
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('VNode vs No-VNode Comparison', fontsize=14)
    
    for ax, (metric, title) in zip(axes.flat, metrics_to_compare):
        for agg, layers, label, color in [(agg_vn, layers_vn, 'With VNode', 'tab:red'),
                                           (agg_novn, layers_novn, 'Without VNode', 'tab:blue')]:
            if metric in agg:
                mean = agg[metric]['mean']
                std = agg[metric]['std']
                valid_mask = ~np.isnan(mean)
                vl = np.array(layers)[valid_mask]
                ax.plot(vl, mean[valid_mask], color=color, linewidth=2, marker='o', markersize=3, label=label)
                ax.fill_between(vl, (mean - std)[valid_mask], (mean + std)[valid_mask], alpha=0.15, color=color)
        ax.set_xlabel('Layer')
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/figure3_vnode_comparison.pdf', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print("Missing diagnostics for VNode or NoVNode experiment. Run training first.")

## 4. Depth comparison

In [ ]:
depth_configs = {
    '4 layers': 'zinc-vnode-rwse-4L',
    '10 layers': 'zinc-vnode-rwse-10L',
    '16 layers': 'zinc-vnode-rwse-16L',
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for label, eid in depth_configs.items():
    diag_path = f'../outputs/{eid}/diagnostics.pkl'
    if os.path.exists(diag_path):
        metrics = load_diagnostics(diag_path)
        agg, layers = aggregate_metrics(metrics)
        
        for ax, metric in zip(axes, ['max_sink_score', 'matrix_entropy']):
            if metric in agg:
                mean = agg[metric]['mean']
                valid_mask = ~np.isnan(mean)
                vl = np.array(layers)[valid_mask]
                ax.plot(vl, mean[valid_mask], linewidth=2, marker='o', markersize=3, label=label)

for ax, title in zip(axes, ['Max Sink Score vs Depth', 'Matrix Entropy vs Depth']):
    ax.set_xlabel('Layer')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Figure 3b: Depth Effect on Sink Formation (H7)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figure3b_depth_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show()